# Модуль 28. Обработка данных с Polars. Домашнее задание (HW)

Данные:
- Файл: NF-CSE-CIC-IDS2018-V2.parquet.
- Описание: датасет сетевого трафика с метками Label (бинарная метка: 0 = "Benign", 1 = "Attack") и Attack (строковая метка типа атаки, например: "SSH-Bruteforce", "DDoS").
- Размер: ~17 миллионов строк, 43 признака.

Используемые колонки:
- Label: бинарная метка (0 = норма, 1 = атака).
- Attack: строковая метка типа атаки (например, "SSH-Bruteforce").
- PROTOCOL: тип протокола (TCP = 6, UDP = 17 и т. д.).
- IN_BYTES, OUT_BYTES, FLOW_DURATION_MILLISECONDS и др.

#### 1. Загрузка и первичный анализ

In [1]:
import polars as pl

#Загрузите файл NF-CSE-CIC-IDS2018-V2.parquet в Polars DataFrame.
df = pl.read_parquet(r'data\NF-CSE-CIC-IDS2018-V2.parquet')
lazy_df = df.lazy()

In [2]:
#Выведите:

#форму датасета (количество строк и колонок);
print(f"Форма датасета: {df.shape}")

#типы колонок (последние пять);
print(f"\nТипы последних пятни колонок:\n{df.select(df.columns[-5:]).schema}")
###print(df.select(df.columns[-5:]).glimpse())
###print(df.dtypes[-5:])

#уникальные значения в колонке Label;
print(f"\nУникальные значения в колонке Label:\n{df.select(
    pl.col('Label').unique())
    }")

#уникальные значения в колонке Attack.
print(f"\nУникальные значения в колонке Attack:\n{df.select(
    pl.col('Attack').unique())
}")

Форма датасета: (17129715, 43)

Типы последних пятни колонок:
Schema({'DNS_QUERY_TYPE': Int16, 'DNS_TTL_ANSWER': Int32, 'FTP_COMMAND_RET_CODE': Int8, 'Label': Int8, 'Attack': String})

Уникальные значения в колонке Label:
shape: (2, 1)
┌───────┐
│ Label │
│ ---   │
│ i8    │
╞═══════╡
│ 0     │
│ 1     │
└───────┘

Уникальные значения в колонке Attack:
shape: (15, 1)
┌──────────────────────────┐
│ Attack                   │
│ ---                      │
│ str                      │
╞══════════════════════════╡
│ DDOS attack-LOIC-UDP     │
│ DoS attacks-GoldenEye    │
│ Brute Force -Web         │
│ DDoS attacks-LOIC-HTTP   │
│ DoS attacks-Hulk         │
│ …                        │
│ SQL Injection            │
│ DDOS attack-HOIC         │
│ Infilteration            │
│ Bot                      │
│ DoS attacks-SlowHTTPTest │
└──────────────────────────┘


#### 2. Распределение по меткам:

Подсчитайте:
- сколько записей помечено как Label == 0 ("Benign");
- сколько записей помечено как Label == 1 ("Attack").
- Выведите процентное соотношение.

In [3]:
result2 = df.group_by('Label').agg(
     pl.len().alias('count'),
    (pl.len() / df.height * 100).alias('percentage')
)
print(result2)

#Простой вариант. Работает очень медленно.
#total = df.height
#benign_count = df.filter(pl.col('Label') == 0).height
#attack_count = df.filter(pl.col('Label') == 1).height
#print(f"Benign (Label == 0): {benign_count} записей ({(benign_count / total) * 100} %)")
#print(f"Attack (Label == 1): {attack_count} записей ({(attack_count / total) * 100} %)")

shape: (2, 3)
┌───────┬──────────┬────────────┐
│ Label ┆ count    ┆ percentage │
│ ---   ┆ ---      ┆ ---        │
│ i8    ┆ u32      ┆ f64        │
╞═══════╪══════════╪════════════╡
│ 0     ┆ 15101685 ┆ 88.160749  │
│ 1     ┆ 2028030  ┆ 11.839251  │
└───────┴──────────┴────────────┘


#### 3. Создание бинарного признака:

Добавьте колонку is_attack, которая:
- 1, если Label != 0 (атака);
- 0, если Label == 0 (норма).

In [4]:
df = df.with_columns(
    pl.when(pl.col('Label') != 0)
    .then(1)
    .otherwise(0)
    .alias('is_attack')
)

lazy_df = df.lazy()

print(df.select(pl.col('Label'),pl.col('is_attack')).head(5))

shape: (5, 2)
┌───────┬───────────┐
│ Label ┆ is_attack │
│ ---   ┆ ---       │
│ i8    ┆ i32       │
╞═══════╪═══════════╡
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
└───────┴───────────┘


#### 4. Агрегация по типам атак:
- Отфильтруйте только атаки (Label != 0).
- Сгруппируйте данные по колонке Attack.
- Посчитайте:
- среднюю длительность потока (FLOW_DURATION_MILLISECONDS);
- среднее количество входящих байт (IN_BYTES);
- общее количество записей для каждого типа атаки.
- Отсортируйте по убыванию avg_in_bytes.

In [5]:
result4 = (
    lazy_df
    .filter(pl.col('Label') !=0)
    .group_by('Attack')
    .agg(pl.mean('FLOW_DURATION_MILLISECONDS').alias('avg_in_ms'),
         pl.mean('IN_BYTES').alias('avg_in_bytes'),
         pl.len().alias('count')
         )
    .sort('avg_in_bytes', descending=True)
).collect()

#Записываем результат в файл
result4.write_parquet(r'data\attack_summary_by_type.parquet',compression='zstd')

print(result4)

shape: (14, 4)
┌──────────────────────────┬───────────────┬──────────────┬─────────┐
│ Attack                   ┆ avg_in_ms     ┆ avg_in_bytes ┆ count   │
│ ---                      ┆ ---           ┆ ---          ┆ ---     │
│ str                      ┆ f64           ┆ f64          ┆ u32     │
╞══════════════════════════╪═══════════════╪══════════════╪═════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6      ┆ 5.8540e6     ┆ 2112    │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6      ┆ 25991.904925 ┆ 207078  │
│ Brute Force -XSS         ┆ 3.7568e6      ┆ 16871.281553 ┆ 927     │
│ Brute Force -Web         ┆ 3.7553e6      ┆ 9797.653756  ┆ 2143    │
│ SSH-Bruteforce           ┆ 733291.768981 ┆ 5828.140747  ┆ 94979   │
│ …                        ┆ …             ┆ …            ┆ …       │
│ Infilteration            ┆ 234910.506514 ┆ 791.858613   ┆ 115513  │
│ Bot                      ┆ 1.2340e6      ┆ 677.354225   ┆ 28033   │
│ SQL Injection            ┆ 3.7282e6      ┆ 599.328704   ┆ 432     │
│ DDO

##### 5. Топ-3 атак по трафику:
- Выведите топ-3 типа атак по среднему объёму входящего трафика (avg_in_bytes).

In [6]:
#Выбираем первые три из ответа предыдущего задания
print(result4.select(pl.col('Attack')).head(3))

#Вариант через через top_k
#print(result4.top_k(3, by=pl.col('avg_in_bytes')).select(pl.col('Attack')))

shape: (3, 1)
┌────────────────────────┐
│ Attack                 │
│ ---                    │
│ str                    │
╞════════════════════════╡
│ DDOS attack-LOIC-UDP   │
│ DDoS attacks-LOIC-HTTP │
│ Brute Force -XSS       │
└────────────────────────┘


#### 6. Распределение по протоколам:
Выведите:
- общее распределение по PROTOCOL;
- распределение по PROTOCOL только для Benign;
- распределение по PROTOCOL только для Attack (с разбивкой по Attack);
- сравнение PROTOCOL между Benign и Attack.

In [7]:
result6_1 = (
    lazy_df
    .group_by(pl.col('PROTOCOL'))
    .agg(pl.len().alias('count'))
    .with_columns(
        (('count' / pl.sum('count') * 100))
        .round(5)
        .alias('percentage')
    )
    .sort('count',descending=True)
).collect()


result6_2 = (
    lazy_df
    .filter(pl.col('Label') == 0)
    .group_by(pl.col('PROTOCOL'))
    .agg(pl.len().alias('benign_count'))
    .with_columns(
        (('benign_count') / pl.sum('benign_count') * 100)
        .round(5)
        .alias('percentage')
    )
    .sort('benign_count', descending=True)
).collect()

result6_3_1 = (
    lazy_df
    .filter(pl.col('Label') == 1)
    .group_by(pl.col('PROTOCOL'))
    .agg(pl.len().alias('attack_count'))
    .with_columns(
        (('attack_count') / pl.sum('attack_count') * 100)
        .round(5)
        .alias('percentage')
    )
    .sort('attack_count', descending=True)
).collect()

result6_3_2 = (
    lazy_df
    .filter(pl.col('Label') == 1)
    .group_by(pl.col('PROTOCOL'),pl.col('Attack'))
    .agg(pl.len().alias('attack_count'))
    .with_columns(
        (('attack_count') / pl.sum('attack_count') * 100)
        .round(5)
        .alias('percentage')
    )
    .sort('PROTOCOL','attack_count', descending=[False,True])
).collect()

#Можно было бы выполнить join on='PROTOCOL' предыдущих результатов, но мы пойдем другим путём
result6_4 = lazy_df.with_columns(
    pl.when(pl.col('Label') == 0).then(pl.col('PROTOCOL')).otherwise(None).alias('b'),
    pl.when(pl.col('Label') == 1).then(pl.col('PROTOCOL')).otherwise(None).alias('a')
).group_by('PROTOCOL').agg([
    pl.col('b').count().alias('benign'),
    pl.col('a').count().alias('attack')
]).with_columns(
    (pl.col('benign') / (pl.col('benign') + pl.col('attack')) * 100).round(2).alias('benign_%'),
    (pl.col('attack') / (pl.col('benign') + pl.col('attack')) * 100).round(2).alias('attack_%')
).sort('PROTOCOL').collect()

print(f"Общее распределение по PROTOCOL:\n{result6_1}")
print(f"Распределение по PROTOCOL только для Benign:\n{result6_2}")
print(f"Распределение по PROTOCOL только для Attack :\n{result6_3_1}")
# остальное перенесено в следую ячейку, что бы избежать обрезания вывода и появления области прокрутки

Общее распределение по PROTOCOL:
shape: (6, 3)
┌──────────┬─────────┬────────────┐
│ PROTOCOL ┆ count   ┆ percentage │
│ ---      ┆ ---     ┆ ---        │
│ i8       ┆ u32     ┆ f64        │
╞══════════╪═════════╪════════════╡
│ 6        ┆ 9346287 ┆ 54.56184   │
│ 17       ┆ 7776756 ┆ 45.39921   │
│ 1        ┆ 4857    ┆ 0.02835    │
│ 2        ┆ 976     ┆ 0.0057     │
│ 58       ┆ 836     ┆ 0.00488    │
│ 47       ┆ 3       ┆ 0.00002    │
└──────────┴─────────┴────────────┘
Распределение по PROTOCOL только для Benign:
shape: (6, 3)
┌──────────┬──────────────┬────────────┐
│ PROTOCOL ┆ benign_count ┆ percentage │
│ ---      ┆ ---          ┆ ---        │
│ i8       ┆ u32          ┆ f64        │
╞══════════╪══════════════╪════════════╡
│ 17       ┆ 7688529      ┆ 50.91173   │
│ 6        ┆ 7406897      ┆ 49.04682   │
│ 1        ┆ 4537         ┆ 0.03004    │
│ 2        ┆ 883          ┆ 0.00585    │
│ 58       ┆ 836          ┆ 0.00554    │
│ 47       ┆ 3            ┆ 0.00002    │
└──────────

In [8]:
print(f"Распределение по PROTOCOL только для Attack (с разбивкой по Attack):\n{result6_3_2}")
print(f"Сравнение PROTOCOL между Benign и Attack:\n{result6_4}")

Распределение по PROTOCOL только для Attack (с разбивкой по Attack):
shape: (20, 4)
┌──────────┬────────────────────────┬──────────────┬────────────┐
│ PROTOCOL ┆ Attack                 ┆ attack_count ┆ percentage │
│ ---      ┆ ---                    ┆ ---          ┆ ---        │
│ i8       ┆ str                    ┆ u32          ┆ f64        │
╞══════════╪════════════════════════╪══════════════╪════════════╡
│ 1        ┆ Infilteration          ┆ 320          ┆ 0.01578    │
│ 2        ┆ Infilteration          ┆ 93           ┆ 0.00459    │
│ 6        ┆ DDOS attack-HOIC       ┆ 1066881      ┆ 52.60677   │
│ 6        ┆ DoS attacks-Hulk       ┆ 432648       ┆ 21.33341   │
│ 6        ┆ DDoS attacks-LOIC-HTTP ┆ 189750       ┆ 9.35637    │
│ …        ┆ …                      ┆ …            ┆ …          │
│ 17       ┆ Infilteration          ┆ 68676        ┆ 3.38634    │
│ 17       ┆ DDoS attacks-LOIC-HTTP ┆ 17328        ┆ 0.85443    │
│ 17       ┆ DDOS attack-LOIC-UDP   ┆ 2112         ┆ 0.104

# 7. Сравнение метрик: Benign vs Attack:
Сгруппируйте по is_attack и посчитайте:
- средние значения IN_BYTES, OUT_BYTES, FLOW_DURATION_MILLISECONDS;
- общее количество записей.

In [9]:
result7 = (
    lazy_df    
    .group_by(pl.col('is_attack'))
    .agg(pl.mean('IN_BYTES').alias('avg_IN_BYTES'),
         pl.mean('OUT_BYTES').alias('avg_OUT_BYTES'),
         pl.mean('FLOW_DURATION_MILLISECONDS').alias('avg_FLOW_DURATION_MILLISECONDS'),
         pl.len().alias('count')
         )    
    .sort('is_attack', descending=False)
).collect()

print(result7)

shape: (2, 5)
┌───────────┬──────────────┬───────────────┬────────────────────────────────┬──────────┐
│ is_attack ┆ avg_IN_BYTES ┆ avg_OUT_BYTES ┆ avg_FLOW_DURATION_MILLISECONDS ┆ count    │
│ ---       ┆ ---          ┆ ---           ┆ ---                            ┆ ---      │
│ i32       ┆ f64          ┆ f64           ┆ f64                            ┆ u32      │
╞═══════════╪══════════════╪═══════════════╪════════════════════════════════╪══════════╡
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107                  ┆ 15101685 │
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6                       ┆ 2028030  │
└───────────┴──────────────┴───────────────┴────────────────────────────────┴──────────┘


# 8. (Бонус) Эвристика детектирования:

Создайте колонку is_suspicious, которая равна:
- 1, если:
    - bytes_ratio > 10 (входящий трафик >> исходящего);
    - FLOW_DURATION_MILLISECONDS < 500 (короткий поток);
    - IN_PKTS > 10 (много пакетов).
- 0 — в противном случае.

Подсчитайте:
- сколько записей помечено как is_suspicious == 1;
- сколько из них на самом деле являются атаками (Label != 0).
- Вычислите точность эвристики: true_attacks / total_flagged.

Формулы, которые могут понадобиться:
- bytes_ratio = IN_BYTES / (OUT_BYTES + 1) — избегаем деления на ноль.
- total_bytes = IN_BYTES + OUT_BYTES.
- packet_size_avg = total_bytes / (IN_PKTS + OUT_PKTS + 1).

In [10]:
#Добавление необходимых колонок в датафрейм
df = (
    lazy_df
    .with_columns(
        #добавляем колонку bytes_ratio
        (pl.col('IN_BYTES') / (pl.col('OUT_BYTES') + 1)).alias('bytes_ratio'),
        #добавляем колонку total_bytes
        (pl.col('IN_BYTES') + (pl.col('OUT_BYTES'))).alias('total_bytes')        
    )
    .with_columns(
        #добавляем колонку packet_size_avg
        (pl.col('total_bytes') / (pl.col('IN_PKTS') + pl.col('OUT_PKTS') + 1)).alias('packet_size_avg')
    )
    .with_columns(        
        pl.when(
            (pl.col('bytes_ratio') > 10) &
            (pl.col('FLOW_DURATION_MILLISECONDS') < 500) &
            (pl.col('IN_PKTS') > 10)
        )
        .then(1)
        .otherwise(0)
        .alias('is_suspicious')
    )
).collect()

# Подсчёт общего числа подозрительных записей
is_suspicious_total = df.filter(pl.col('is_suspicious') == 1).height

# Подсчёт числа подозрительных записей, которые являются атаками
correct_suspicions = df.filter(
    (pl.col('is_suspicious') == 1) &
    (pl.col('Label') != 0)
).height

# Вычисление точности эвристики
precision = correct_suspicions / is_suspicious_total

#Вывод результатов
print(f"Датафрейм с добавленными колонками bytes_ratio, total_bytes, packet_size_avg:\n{df}")
print(f"Всего помечено как подозрительные: {is_suspicious_total}")
print(f"Из них являются атаками: {correct_suspicions}")
print(f"Точность эвристики: {precision:.4f} ({precision * 100:.2f}%)")


Датафрейм с добавленными колонками bytes_ratio, total_bytes, packet_size_avg:
shape: (17_129_715, 48)
┌───────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ L4_SRC_PO ┆ L4_DST_PO ┆ PROTOCOL ┆ L7_PROTO  ┆ … ┆ bytes_rat ┆ total_byt ┆ packet_si ┆ is_suspic │
│ RT        ┆ RT        ┆ ---      ┆ ---       ┆   ┆ io        ┆ es        ┆ ze_avg    ┆ ious      │
│ ---       ┆ ---       ┆ i8       ┆ f32       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ i32       ┆ i32       ┆          ┆           ┆   ┆ f64       ┆ i32       ┆ f64       ┆ i32       │
╞═══════════╪═══════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 40894     ┆ 22        ┆ 6        ┆ 92.0      ┆ … ┆ 0.840149  ┆ 6929      ┆ 153.97777 ┆ 0         │
│           ┆           ┆          ┆           ┆   ┆           ┆           ┆ 8         ┆           │
│ 29622     ┆ 3389      ┆ 6        ┆ 0.0       ┆ … ┆ 0.94439   ┆ 3950      ┆ 151.92307 ┆ 0